# Asesor de Hipoteca Personalizada — Pricing Provincial Explicable

**Modelo de pricing hipotecario personalizado con explicabilidad AI Act**

Este notebook calcula un tipo de interes personalizado para cada cliente en funcion de:
- Variables macroeconomicas de su **provincia** (Precio_m2, PIB_pc, Tasa_Paro, IPC, Euribor, tasa_ahorro)
- **Perfil del cliente** (ingresos vs PIB per capita provincial, edad, ahorros)

## Modelo: Gradient Boosting + SHAP
- **Gradient Boosting Regressor** para predecir el *Spread Hipotecario* (diferencial entre tipo hipotecario y Euribor)
- **SHAP (SHapley Additive exPlanations)** para explicar cada prediccion al cliente/gestor de forma transparente
- Cumple con los requisitos de **explicabilidad de la AI Act** para modelos de credito bancario

## Indice
1. Configuracion del cliente (inputs)
2. Carga de datos y contexto macroeconomico
3. Calculo de impuestos y gastos de compra (por CCAA)
4. Verificacion de ahorros suficientes
5. Modelo de Pricing: comparativa, seleccion y justificacion
6. Prediccion del Spread y tipo de interes personalizado
7. Explicabilidad SHAP — por que este tipo
8. Cuota mensual y ratio DTI
9. Decision bancaria (3 niveles)
10. Resultado final

## 1. Configuracion del cliente
Editar los parametros de abajo para simular distintos perfiles.

> **Nota**: El notebook soporta años historicos (2005-2025) y futuros (2026-2030).
> Para años futuros, se utilizan las **proyecciones del BCE** para Euribor e IPC,
> y las **predicciones provinciales SARIMA** para el resto de variables macro.

In [ ]:
# ═══════════════════════════════════════════════════════
#  PARAMETROS DEL CLIENTE — EDITAR AQUI
# ═══════════════════════════════════════════════════════

year         = '2023'        # Año de referencia: '2005'-'2025' (historico) o '2026'-'2030' (futuro)
tipo         = 'variable'    # 'variable' o 'fijo'
province     = 'Valladolid'  # Provincia (debe existir en el dataset)
income       = 1300.0        # Ingresos mensuales netos (EUR)
savings      = 35_000.0      # Ahorros brutos disponibles (EUR)
age          = 30            # Edad del comprador
other_debts  = 0.0           # Otras deudas fijas mensuales (EUR)
requested_years = 25           # Plazo deseado (años): 5-30
price_mode   = 'auto'        # 'auto' = precio medio 80m2 | 'manual' = custom_price
custom_price = 0.0           # Solo si price_mode = 'manual'

print(f"Cliente: {age} años | Ingresos: {income:,.0f} EUR/mes | Ahorros: {savings:,.0f} EUR")
print(f"Provincia: {province} | Año: {year} | Hipoteca: {tipo}")

## 2. Carga de datos y contexto macroeconomico

Cargamos tres fuentes:
- `dataset_final_tfm.csv` — datos trimestrales por provincia (52 provincias, 2005-2025)
- `tipos_hipotecas.csv` — APRC y Euribor trimestrales del BCE/BdE
- `predicciones_provinciales_2026_2030.csv` — proyecciones provinciales SARIMA para 2025-2030

### Proyecciones futuras (2026-2030)

| Variable | Fuente |
|---|---|
| **Euribor** | Proyecciones BCE (exogenas) |
| **IPC** | Proyecciones BCE (exogenas) |
| **PIB_pc, Tasa_Paro, Precio_m2, tasa_ahorro** | SARIMA provincial (endogenas) |
| **APRC** | Euribor BCE + Diferencial SARIMAX |

El **spread real** (diferencial) = APRC - Euribor. Es la variable que nuestro modelo predice.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATASET_PATH   = 'dataset_final_tfm.csv'
TIPOS_PATH     = 'Web/static/tipos_hipotecas.csv'
PROV_PRED_PATH = 'predicciones_provinciales_2026_2030.csv'

df       = pd.read_csv(DATASET_PATH)
df_tipos = pd.read_csv(TIPOS_PATH)
df_tipos['year'] = df_tipos['TIME_PERIOD'].str[:4]

# Cargar predicciones provinciales
try:
    df_prov_pred = pd.read_csv(PROV_PRED_PATH)
    df_prov_pred['year'] = df_prov_pred['trimestre'].str[:4]
    has_prov_pred = True
    print(f"Predicciones provinciales: {len(df_prov_pred):,} filas | {df_prov_pred['provincia'].nunique()} provincias")
except:
    df_prov_pred = pd.DataFrame()
    has_prov_pred = False
    print("AVISO: No se encontro el fichero de predicciones provinciales")

# ── Proyecciones BCE (exogenas) — fuente: BCE Staff Projections ──
ESCENARIOS_BCE = {
    'Euribor': {'2026': 2.00, '2027': 2.10, '2028': 2.30, '2029': 2.15, '2030': 2.15},
    'IPC':     {'2026': 1.88, '2027': 2.10, '2028': 2.00, '2029': 2.00, '2030': 2.00},
}

# ── Diferencial futuro (APRC - Euribor) via SARIMAX ──
def calcular_diferencial_futuro():
    import statsmodels.api as sm
    dt = df_tipos.copy()
    dt['y'] = dt['TIME_PERIOD'].str[:4]
    ann = dt.groupby('y')['Diferencia'].mean().dropna()
    ts = ann.values
    try:
        model = sm.tsa.statespace.SARIMAX(ts, order=(1,1,1),
                    enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        last_yr = int(ann.index[-1])
        steps = max(0, 2031 - last_yr)
        fc = res.forecast(steps=steps)
        return {str(last_yr+1+i): float(v) for i, v in enumerate(fc)}
    except:
        return {'2026': 0.50, '2027': 0.55, '2028': 0.55, '2029': 0.55, '2030': 0.55}

diff_futuro = calcular_diferencial_futuro()

year_int = int(year)
is_future = year_int > 2025

print(f"\nDataset: {df.shape[0]:,} filas | {df['Provincia'].nunique()} provincias | {df['Trimestre'].min()[:4]}-{df['Trimestre'].max()[:4]}")
print(f"Año seleccionado: {year} ({'FUTURO — usando proyecciones' if is_future else 'HISTORICO'})")

In [ ]:
# ═══════════════════════════════════════════════════════
#  OBTENER VALORES MACRO PARA EL AÑO SELECCIONADO
# ═══════════════════════════════════════════════════════

if is_future:
    # ── Euribor y APRC desde proyecciones BCE + diferencial SARIMAX ──
    euribor = ESCENARIOS_BCE['Euribor'].get(year, 2.15)
    dif_proj = diff_futuro.get(year, 0.55)
    aprc = round(euribor + dif_proj, 4)
    diferencia = dif_proj

    # ── Macro provincial desde predicciones SARIMA ──
    ipc = ESCENARIOS_BCE['IPC'].get(year, 2.0)

    if has_prov_pred:
        pp_yr = df_prov_pred[df_prov_pred['year'] == year]
        pp_prov = pp_yr[pp_yr['provincia'] == province]

        # Intentar extraer valores provinciales
        def get_prov_value(var_name, fallback):
            vals = pp_prov[pp_prov['variable'] == var_name]['valor']
            return float(vals.mean()) if not vals.empty else fallback

        # Fallbacks = ultimo año real del dataset
        last_real = df[df['Trimestre'].str.startswith('2025')]
        last_prov = last_real[last_real['Provincia'] == province]
        fb_pm2 = float(last_prov['Precio_m2'].mean()) if not last_prov.empty else float(last_real['Precio_m2'].mean())
        fb_pib = float(last_prov['PIB_pc'].mean()) if not last_prov.empty and not last_prov['PIB_pc'].isna().all() else 25000
        fb_paro = float(last_prov['Tasa_Paro'].mean()) if not last_prov.empty else 12.0
        fb_ahorro = float(last_real['tasa_ahorro'].mean()) if not last_real.empty and not last_real['tasa_ahorro'].isna().all() else 8.0

        precio_m2 = get_prov_value('Precio_m2', fb_pm2)
        pib_pc_prov = get_prov_value('PIB_pc', fb_pib)
        tasa_paro_prov = get_prov_value('Tasa_Paro', fb_paro)
        tasa_ah = get_prov_value('tasa_ahorro', fb_ahorro)
    else:
        # Sin fichero provincial — usar ultimoCONOCIDO
        last_real = df[df['Trimestre'].str.startswith('2025')]
        last_prov = last_real[last_real['Provincia'] == province]
        precio_m2 = float(last_prov['Precio_m2'].mean()) if not last_prov.empty else float(last_real['Precio_m2'].mean())
        pib_pc_prov = float(last_prov['PIB_pc'].mean()) if not last_prov.empty and not last_prov['PIB_pc'].isna().all() else 25000
        tasa_paro_prov = float(last_prov['Tasa_Paro'].mean()) if not last_prov.empty else 12.0
        tasa_ah = float(last_real['tasa_ahorro'].mean()) if not last_real.empty else 8.0

    print(f"Fuente para {year}: PROYECCIONES")
    print(f"  Euribor BCE: {euribor:.3f}% | APRC proyectado: {aprc:.3f}% | Diferencial SARIMAX: {diferencia:.3f}%")
    print(f"  IPC BCE: {ipc:.2f}%")
    print(f"  Precio_m2 provincial: {precio_m2:,.2f} EUR/m2")
    print(f"  PIB_pc provincial: {pib_pc_prov:,.0f} EUR")
    print(f"  Tasa_Paro provincial: {tasa_paro_prov:.2f}%")
    print(f"  Tasa ahorro: {tasa_ah:.2f}%")

else:
    # ── Datos historicos del dataset ──
    ydf = df[df['Trimestre'].str.startswith(year)].copy()

    ytipos = df_tipos[df_tipos['year'] == year]
    if not ytipos.empty:
        euribor    = float(ytipos['EURIBOR_VALUE'].mean())
        aprc       = float(ytipos['APRC_VALUE'].mean())
        diferencia = float(ytipos['Diferencia'].mean())
    else:
        euribor = float(ydf['Euribor'].mean())
        aprc = euribor
        diferencia = 0.0

    macro      = ydf[['IPC','Tasa_Paro','PIB_pc','tasa_ahorro']].mean()
    ipc        = float(macro['IPC'])
    tasa_paro  = float(macro['Tasa_Paro'])
    pib_pc     = float(macro['PIB_pc'])
    tasa_ah    = float(macro['tasa_ahorro'])

    precio_m2 = float(ydf['Precio_m2'].mean())
    prov_data = ydf[ydf['Provincia'] == province]
    if not prov_data.empty:
        precio_m2 = float(prov_data['Precio_m2'].mean())
        pib_pc_prov = float(prov_data['PIB_pc'].mean()) if not prov_data['PIB_pc'].isna().all() else pib_pc
        tasa_paro_prov = float(prov_data['Tasa_Paro'].mean())
    else:
        pib_pc_prov = pib_pc
        tasa_paro_prov = tasa_paro

    print(f"Fuente para {year}: DATOS HISTORICOS")
    print(f"  Euribor: {euribor:.3f}% | APRC: {aprc:.3f}% | Diferencial: {diferencia:.3f}%")
    print(f"  IPC: {ipc:.2f}% | Paro provincial: {tasa_paro_prov:.2f}% | PIB/pc provincial: {pib_pc_prov:,.0f} EUR")
    print(f"  Precio m2 ({province}): {precio_m2:.2f} EUR/m2")

## 3. Calculo de impuestos y gastos de compra (por CCAA)

| Gasto | 2a mano | Nueva |
|---|---|---|
| Impuesto transmision | **ITP** (4-10% segun CCAA) | **IVA** (10%) |
| Acto juridico documentado | — | **AJD** (0.5-1.5% segun CCAA) |
| Notaria | Arancel por tramo de precio | igual |
| Registro | Arancel por tramo de precio | igual |

Tipos extraidos por web scraping de `financialtools.web.app`.

In [ ]:
PROVINCE_TO_CCAA = {
    'Almeria':'Andalucia','Cadiz':'Andalucia','Cordoba':'Andalucia',
    'Granada':'Andalucia','Huelva':'Andalucia','Jaen':'Andalucia',
    'Malaga':'Andalucia','Sevilla':'Andalucia',
    'Huesca':'Aragon','Teruel':'Aragon','Zaragoza':'Aragon',
    'Asturias':'Asturias','Principado de Asturias':'Asturias',
    'Baleares':'Baleares','Illes Balears':'Baleares','Islas Baleares':'Baleares',
    'Las Palmas':'Canarias','Santa Cruz de Tenerife':'Canarias',
    'Cantabria':'Cantabria',
    'Albacete':'Castilla-La Mancha','Ciudad Real':'Castilla-La Mancha',
    'Cuenca':'Castilla-La Mancha','Guadalajara':'Castilla-La Mancha',
    'Toledo':'Castilla-La Mancha',
    'Avila':'Castilla y Leon','Ávila':'Castilla y Leon',
    'Burgos':'Castilla y Leon','Leon':'Castilla y Leon',
    'León':'Castilla y Leon','Palencia':'Castilla y Leon',
    'Salamanca':'Castilla y Leon','Segovia':'Castilla y Leon',
    'Soria':'Castilla y Leon','Valladolid':'Castilla y Leon',
    'Zamora':'Castilla y Leon',
    'Barcelona':'Cataluna','Girona':'Cataluna',
    'Lleida':'Cataluna','Tarragona':'Cataluna',
    'Alicante':'Comunidad Valenciana','Castellon':'Comunidad Valenciana',
    'Castellón':'Comunidad Valenciana',
    'Valencia':'Comunidad Valenciana',
    'Badajoz':'Extremadura','Caceres':'Extremadura','Cáceres':'Extremadura',
    'A Coruña':'Galicia','La Coruna':'Galicia','Lugo':'Galicia',
    'Ourense':'Galicia','Pontevedra':'Galicia',
    'Madrid':'Madrid','Comunidad de Madrid':'Madrid',
    'Murcia':'Murcia','Región de Murcia':'Murcia',
    'Navarra':'Navarra','Comunidad Foral de Navarra':'Navarra',
    'Alava':'Pais Vasco','Álava':'Pais Vasco',
    'Guipuzcoa':'Pais Vasco','Guipúzcoa':'Pais Vasco',
    'Vizcaya':'Pais Vasco',
    'La Rioja':'La Rioja','Ceuta':'Ceuta','Melilla':'Melilla',
}

ITP_GENERAL = {
    'Andalucia':7.0,'Aragon':8.0,'Asturias':8.0,'Baleares':8.0,
    'Canarias':6.5,'Cantabria':10.0,'Castilla-La Mancha':9.0,
    'Castilla y Leon':8.0,'Cataluna':10.0,'Comunidad Valenciana':10.0,
    'Extremadura':8.0,'Galicia':10.0,'Madrid':6.0,'Murcia':8.0,
    'Navarra':6.0,'Pais Vasco':4.0,'La Rioja':7.0,'Ceuta':6.0,'Melilla':6.0,
}

ITP_JOVEN = {
    'Andalucia':3.5,'Aragon':5.0,'Asturias':6.0,'Baleares':6.0,
    'Canarias':5.0,'Cantabria':4.0,'Castilla-La Mancha':5.0,
    'Castilla y Leon':4.0,'Cataluna':5.0,'Comunidad Valenciana':6.0,
    'Extremadura':6.0,'Galicia':5.0,'Madrid':6.0,'Murcia':3.0,
    'Navarra':5.0,'Pais Vasco':4.0,'La Rioja':4.0,'Ceuta':6.0,'Melilla':6.0,
}

AJD = {
    'Andalucia':1.2,'Aragon':1.5,'Asturias':1.2,'Baleares':1.5,
    'Canarias':1.0,'Cantabria':1.5,'Castilla-La Mancha':1.25,
    'Castilla y Leon':1.5,'Cataluna':1.5,'Comunidad Valenciana':1.5,
    'Extremadura':1.2,'Galicia':1.5,'Madrid':0.75,'Murcia':1.5,
    'Navarra':0.5,'Pais Vasco':0.5,'La Rioja':1.0,'Ceuta':0.5,'Melilla':0.5,
}

NOTARIA  = {25000:600,50000:625,100000:675,150000:710,200000:750,250000:790,300000:830,400000:850}
REGISTRO = {25000:395,50000:420,100000:445,150000:470,200000:510,250000:550,300000:590,400000:630}

def get_ccaa(prov):
    if prov in PROVINCE_TO_CCAA:
        return PROVINCE_TO_CCAA[prov]
    for k,v in PROVINCE_TO_CCAA.items():
        if k.lower() in prov.lower() or prov.lower() in k.lower():
            return v
    return ''

def get_aranceles(precio):
    brackets = sorted(NOTARIA.keys())
    idx = len(brackets)-1
    for i,b in enumerate(brackets):
        if precio <= b:
            idx = i; break
    return NOTARIA[brackets[idx]], REGISTRO[brackets[idx]]

def calcular_impuestos(precio, ccaa, edad, nueva=False):
    notaria, registro = get_aranceles(precio)
    joven = edad <= 35
    if nueva:
        iva = precio * 10 / 100
        ajd = precio * AJD.get(ccaa,1.5) / 100
        return dict(tipo='IVA+AJD',itp=0,itp_rate=None,iva=iva,ajd=ajd,
                    notaria=notaria,registro=registro,total=iva+ajd+notaria+registro,tipo_joven=False)
    itp_gen  = ITP_GENERAL.get(ccaa,8.0)
    itp_rate = ITP_JOVEN.get(ccaa,itp_gen) if joven else itp_gen
    itp = precio * itp_rate / 100
    return dict(tipo='ITP',itp=itp,itp_rate=itp_rate,iva=0,ajd=0,
                notaria=notaria,registro=registro,total=itp+notaria+registro,
                tipo_joven=joven and itp_rate < itp_gen)

# ── Calcular precio y CCAA ────────────────────────────────────
ccaa = get_ccaa(province)
if price_mode == 'manual' and custom_price > 0:
    estimated_price = custom_price
else:
    estimated_price = precio_m2 * 80

taxes = calcular_impuestos(estimated_price, ccaa, age)
total_taxes = taxes['total']

print(f"CCAA: {ccaa}")
print(f"Precio estimado de la vivienda: {estimated_price:,.0f} EUR")
print()
print(f"Impuestos y gastos de compra estimados ({ccaa})")
print("-" * 48)
if taxes['itp']:
    suf = ' (tipo reducido jovenes)' if taxes['tipo_joven'] else ''
    print(f"  ITP {taxes['itp_rate']:.0f}%{suf}: {taxes['itp']:,.0f} EUR")
print(f"  Notaria (est.): {taxes['notaria']:,.0f} EUR")
print(f"  Registro (est.): {taxes['registro']:,.0f} EUR")
print(f"  Total gastos estimados: {total_taxes:,.0f} EUR")

## 4. Verificacion de ahorros suficientes

El cliente necesita cubrir **dos conceptos** con sus ahorros:
1. **Impuestos y gastos** de la compra (ITP/IVA + notaria + registro)
2. **Entrada minima** = Precio × (100% - % financiacion maxima)

```
Ahorros netos         = Ahorros brutos - Total gastos (impuestos + notaria + registro)
Entrada minima        = Precio × (1 - financiacion_maxima%)
Condicion: Ahorros netos >= Entrada minima
```

In [ ]:
YOUNG_FINANCING = {
    'Andalucia':(95,35),'Aragon':(95,35),'Asturias':(90,35),
    'Baleares':(100,35),'Canarias':(95,35),'Cantabria':(90,35),
    'Castilla-La Mancha':(90,35),'Castilla y Leon':(97,35),
    'Cataluna':(95,35),'Comunidad Valenciana':(95,35),
    'Extremadura':(95,35),'Galicia':(100,40),'Madrid':(95,40),
    'Murcia':(100,35),'Navarra':(90,35),'Pais Vasco':(100,40),
    'La Rioja':(100,35),'Ceuta':(90,35),'Melilla':(90,35),
}

young_max, young_limit = YOUNG_FINANCING.get(ccaa,(80,35))
is_young = age <= young_limit and ccaa in YOUNG_FINANCING
max_fin  = young_max if is_young else 80.0

# Los ahorros netos YA descuentan gastos (impuestos + notaria + registro)
effective_savings = savings - total_taxes
min_down          = estimated_price * (1 - max_fin/100)
loan_amount       = estimated_price * max_fin / 100
savings_ok        = effective_savings >= min_down

if is_young:
    print(f"Cliente joven en {ccaa}: financiacion maxima {max_fin:.0f}% (limite {young_limit} anos)")
else:
    print(f"Financiacion estandar: {max_fin:.0f}%")
print()
print(f"Descomposicion de necesidades de ahorro:")
print(f"  Impuestos + gastos compra:  {total_taxes:>10,.0f} EUR")
print(f"  Entrada minima ({100-max_fin:.0f}%):      {min_down:>10,.0f} EUR")
print(f"  TOTAL necesario:            {total_taxes + min_down:>10,.0f} EUR")
print()
print(f"  Ahorros brutos:             {savings:>10,.0f} EUR")
print(f"  - Gastos compra:            {total_taxes:>10,.0f} EUR")
print(f"  = Ahorros netos:            {effective_savings:>10,.0f} EUR")
print(f"  vs Entrada minima:          {min_down:>10,.0f} EUR")
print()
print(f"Capital a financiar: {loan_amount:,.0f} EUR")
print(f"Ahorros suficientes: {'SI' if savings_ok else 'NO — faltan {:,.0f} EUR'.format(min_down - effective_savings)}") 

## 5. Modelo de Pricing — Comparativa, seleccion y justificacion

Comparamos **6 modelos** para predecir `Spread_Hipotecas` usando **TimeSeriesSplit CV-5** (validacion temporal correcta que evita data leakage).

### Justificacion de la eleccion del modelo

> **¿Por que Gradient Boosting + SHAP y no Regresion Lineal?**
>
> La eleccion del modelo **no se basa en superar a los lineales en R²**. Con TimeSeriesSplit sobre datos
> donde el spread sufre cambios estructurales (crisis 2008-2012, tipos negativos 2016-2020, subida 2022),
> **ningun modelo generaliza bien hacia adelante** — los R² de CV temporal son negativos o bajos.
>
> La justificacion es diferente:
> 1. **SHAP TreeExplainer** proporciona explicabilidad **exacta** (no aproximada), cumpliendo Art. 13-14 de la AI Act
> 2. En **RMSE / MAE** (metricas de error absoluto), GBR es competitivo o superior
> 3. Captura **no-linealidades** (crisis, cambios de politica monetaria) que los modelos lineales ignoran
> 4. Con hiperparametros conservadores (`max_depth=3`, `min_samples_leaf=5`), el riesgo de sobreajuste es controlado

La tabla siguiente muestra los resultados **reales** del CV temporal:

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

FEATURES = ['Euribor','IPC','Tasa_Paro','Precio_m2','PIB_pc','tasa_ahorro']
TARGET   = 'Spread_Hipotecas'

# Ordenar temporalmente para TimeSeriesSplit
data = df[['Trimestre'] + FEATURES + [TARGET]].dropna().sort_values('Trimestre').reset_index(drop=True)
X = data[FEATURES].values
y = data[TARGET].values

# Usamos Pipeline(Scaler + Modelo) para que el scaler se ajuste en cada fold
# Esto evita data leakage y resuelve overflow numerico
modelos = {
    'Regresion Lineal':    Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge':               Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso':               Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.01, max_iter=5000))]),
    'ElasticNet':          Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000))]),
    'Random Forest':       Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1))]),
    'Gradient Boosting':   Pipeline([('scaler', StandardScaler()), ('model', GradientBoostingRegressor(
                                n_estimators=100, max_depth=3, learning_rate=0.06,
                                min_samples_leaf=5, random_state=42))]),
}

def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    'r2':   'r2',
    'rmse': make_scorer(rmse_scorer, greater_is_better=False),
    'mae':  make_scorer(mean_absolute_error, greater_is_better=False),
}

# TimeSeriesSplit respeta la temporalidad de los datos
tscv = TimeSeriesSplit(n_splits=5)

resultados = []
print(f"{'Modelo':<22} {'R2 CV':>10} {'R2 std':>8} {'RMSE CV':>8} {'MAE CV':>8}")
print('='*60)
for nombre, pipeline in modelos.items():
    cv = cross_validate(pipeline, X, y, cv=tscv, scoring=scoring, n_jobs=-1)
    r2   = cv['test_r2'].mean()
    r2s  = cv['test_r2'].std()
    rmse = -cv['test_rmse'].mean()
    mae  = -cv['test_mae'].mean()
    resultados.append({'Modelo':nombre,'R2_CV':r2,'R2_std':r2s,'RMSE_CV':rmse,'MAE_CV':mae})
    print(f"{nombre:<22} {r2:>10.4f} {r2s:>8.4f} {rmse:>8.4f} {mae:>8.4f}")

df_res = pd.DataFrame(resultados).set_index('Modelo')
print()
print("Nota: R² negativos son normales con TimeSeriesSplit sobre datos con cambios")
print("estructurales. La eleccion se basa en RMSE + explicabilidad SHAP, no en R².")
print()
print(f"Menor RMSE CV: {df_res['RMSE_CV'].idxmin()} ({df_res['RMSE_CV'].min():.4f})")
print(f"Menor MAE CV:  {df_res['MAE_CV'].idxmin()} ({df_res['MAE_CV'].min():.4f})")

In [ ]:
# ── Grafico comparativo ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics = ['RMSE_CV', 'MAE_CV']
titles  = ['RMSE — TimeSeriesSplit CV-5 (menor = mejor)', 'MAE — TimeSeriesSplit CV-5 (menor = mejor)']
highlight = 'Gradient Boosting'

for ax, metric, title in zip(axes, metrics, titles):
    vals   = df_res[metric].values
    labels = df_res.index.tolist()
    colors = ['#27ae60' if l == highlight else '#2c7bb6' for l in labels]
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor='#27ae60', alpha=0.85, label='Modelo elegido (GBR + SHAP = explicabilidad AI Act)'),
                Patch(facecolor='#2c7bb6', alpha=0.85, label='Modelos alternativos')]
fig.legend(handles=legend_elems, loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.08))
fig.suptitle('Error de prediccion — TimeSeriesSplit CV-5 (validacion temporal)', fontweight='bold', fontsize=12)
plt.tight_layout(rect=[0,0.08,1,1])
plt.show()

print("\nNota: Se muestra RMSE y MAE (no R²) porque son las metricas relevantes")
print("cuando hay cambios estructurales. R² de CV temporal es negativo para todos los modelos;")
print("GBR se elige por su capacidad de explicabilidad SHAP, no por superar en R².")

## 6. Tipo de interes personalizado — Gradient Boosting + SHAP

### Proceso:
1. **Entrenar** GBR con hiperparametros conservadores sobre el dataset historico
2. **Evaluar** R² de entrenamiento (referencia) vs R² de CV temporal (validacion real)
3. **Predecir** el spread base para la provincia y año seleccionados
4. **Ajustar** por perfil del cliente (ratio ingresos / PIB_pc provincial)
5. **Calcular tipo final**:
   - **Variable**: `Euribor + spread_ajustado` (Euribor historico o **proyeccion BCE**)
   - **Fijo**: `APRC + ajuste_perfil + prima_fija` (APRC historico o **Euribor BCE + diferencial SARIMAX**)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import shap

# ── 6.1 Entrenar el modelo GBR (hiperparametros conservadores) ──
FEATURES = ['Euribor','IPC','Tasa_Paro','Precio_m2','PIB_pc','tasa_ahorro']
TARGET   = 'Spread_Hipotecas'

data_model = df[FEATURES + [TARGET]].dropna()
scaler_gbr = StandardScaler()
X_all      = scaler_gbr.fit_transform(data_model[FEATURES].values)
y_all      = data_model[TARGET].values

gbr = GradientBoostingRegressor(
    n_estimators=100,       # Conservador (no 200+)
    max_depth=3,            # Arboles poco profundos (max 8 hojas vs 16 con depth=4)
    learning_rate=0.06,     # Lento para evitar sobreajuste
    min_samples_leaf=5,     # Minimo 5 observaciones por hoja
    random_state=42
)
gbr.fit(X_all, y_all)

# Separar claramente train score de CV score
r2_train = gbr.score(X_all, y_all)
r2_cv = df_res.loc['Gradient Boosting', 'R2_CV']
rmse_cv = df_res.loc['Gradient Boosting', 'RMSE_CV']
mae_cv = df_res.loc['Gradient Boosting', 'MAE_CV']

print(f"Modelo GBR entrenado: {len(data_model):,} observaciones")
print(f"  Hiperparametros: n_estimators=100, max_depth=3, lr=0.06, min_samples_leaf=5")
print()
print(f"  R² entrenamiento (referencia, NO es validacion): {r2_train:.4f}")
print(f"  R² CV temporal  (validacion real):                {r2_cv:.4f}")
print(f"  RMSE CV temporal:                                 {rmse_cv:.4f}%")
print(f"  MAE CV temporal:                                  {mae_cv:.4f}%")
print()

# ── 6.2 Predecir spread base provincial ──────────────────────────
client_macro = {
    'Euribor':     euribor,
    'IPC':         ipc,
    'Tasa_Paro':   tasa_paro_prov,
    'Precio_m2':   precio_m2,
    'PIB_pc':      pib_pc_prov,
    'tasa_ahorro': tasa_ah
}

X_client = np.array([client_macro[f] for f in FEATURES]).reshape(1, -1)
X_client_sc = scaler_gbr.transform(X_client)

spread_base = gbr.predict(X_client_sc)[0]
print(f"Spread base predicho para {province} ({year}): {spread_base:.4f}%")

# ── 6.3 Ajuste por perfil del cliente ────────────────────────────
ingreso_anual = income * 12
income_ratio = ingreso_anual / pib_pc_prov if pib_pc_prov > 0 else 1.0

def calcular_ajuste_perfil(income_ratio):
    # Funcion tangente hiperbolica centrada en 1.0
    # income_ratio = 1.0 -> ajuste = 0
    # income_ratio > 1.5 -> reduccion del spread (hasta -0.25%)
    # income_ratio < 0.5 -> incremento del spread (hasta +0.30%)
    x = (income_ratio - 1.0) * 2.0
    adj = -0.25 * np.tanh(x)
    if income_ratio < 0.5:
        adj += 0.05
    return round(adj, 4)

profile_adj = calcular_ajuste_perfil(income_ratio)
spread_final = max(0.0, spread_base + profile_adj)

print(f"Income ratio (ingreso_anual / PIB_pc_prov): {income_ratio:.3f}")
print(f"Ajuste por perfil del cliente: {profile_adj:+.4f}%")
print(f"Spread final ajustado: {spread_final:.4f}%")
print()

# ── 6.4 Tipo de interés final ──────────────────────────────────
if tipo == 'variable':
    rate = round(euribor + spread_final, 2)
    fuente_euribor = 'proyeccion BCE' if is_future else 'historico'
    rate_desc = f"Euribor ({euribor:.3f}%, {fuente_euribor}) + Spread ({spread_final:.3f}%)"
else:
    prima_fija = 0.50
    rate = round(aprc + profile_adj + prima_fija, 2)
    fuente_aprc = 'Euribor BCE + diferencial SARIMAX' if is_future else 'historico BdE'
    rate_desc = f"APRC ({aprc:.3f}%, {fuente_aprc}) + Ajuste ({profile_adj:+.3f}%) + Prima fija ({prima_fija:.2f}%)"

print("=" * 60)
print(f"  TIPO OFRECIDO: {rate:.2f}% TIN ({tipo})")
print(f"  Calculo: {rate_desc}")
if is_future:
    print(f"  Fuente tipos: Proyecciones BCE + SARIMAX ({year})")
print("=" * 60)

## 7. Explicabilidad SHAP — ¿Por que este tipo?

**SHAP (SHapley Additive exPlanations)** descompone la prediccion del spread en la contribucion **exacta** de cada variable macroeconomica. Esto permite:

1. **Al cliente**: entender por que su tipo es mas alto o mas bajo
2. **Al gestor bancario**: justificar la decision ante auditoria
3. **Al regulador (AI Act)**: verificar que no hay discriminacion y que el modelo es transparente

Los valores SHAP suman exactamente la diferencia entre la prediccion y la media del modelo.

In [ ]:
import shap
import matplotlib.pyplot as plt

# ── SHAP TreeExplainer ──────────────────────────────────────────
explainer = shap.TreeExplainer(gbr)
shap_values_client = explainer.shap_values(X_client_sc)
expected_val = float(explainer.expected_value)

# Tabla de contribuciones
print(f"Contribucion SHAP al Spread Hipotecario para {province} ({year}):")
print(f"{'Variable':<14} {'Valor':<14} {'SHAP':>12}")
print("-" * 42)
contribs_shap = []
for i, feat in enumerate(FEATURES):
    val = client_macro[feat]
    shp = shap_values_client[0][i]
    contribs_shap.append({'var': feat, 'val': val, 'shap': shp})
    s = '+' if shp >= 0 else ''
    print(f"{feat:<14} {val:<14.3f} {s}{shp:>10.4f}")
print("-" * 42)
print(f"{'Base value':<14} {'':<14} {expected_val:>10.4f}")
print(f"{'Prediccion':<14} {'':<14} {spread_base:>10.4f}")
print(f"{'+ Ajuste perfil':<14} {'':<14} {profile_adj:>+10.4f}")
print(f"{'= Spread final':<14} {'':<14} {spread_final:>10.4f}")

In [ ]:
# ── Grafico Waterfall SHAP para este cliente ─────────────────────
shap_explanation = shap.Explanation(
    values=shap_values_client[0],
    base_values=float(explainer.expected_value),
    data=X_client_sc[0],
    feature_names=FEATURES
)

fig, ax = plt.subplots(figsize=(10, 5))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'SHAP — Descomposicion del Spread para {province} ({year})', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grafico de barras de contribuciones al spread ────────────────
contribs_shap.sort(key=lambda x: abs(x['shap']), reverse=True)

fig, ax = plt.subplots(figsize=(9, 3.5))
labels = [c['var'] for c in contribs_shap]
vals   = [c['shap'] for c in contribs_shap]
colors = ['#c0392b' if v >= 0 else '#27ae60' for v in vals]
ax.barh(labels, vals, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
for i,(v,_) in enumerate(zip(vals,labels)):
    s = '+' if v>=0 else ''
    ax.text(v+(0.003 if v>=0 else -0.003), i, f'{s}{v:.4f}',
            va='center', ha='left' if v>=0 else 'right', fontsize=9)
ax.set_xlabel('Contribucion SHAP al Spread (%)')
ax.set_title(f'Contribucion de variables al spread — {province} ({year})', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Importancia global SHAP ──────────────────────────────────────
X_sample = X_all[:500] if len(X_all) > 500 else X_all
shap_values_global = explainer.shap_values(X_sample)

fig, ax = plt.subplots(figsize=(10, 5))
shap.summary_plot(shap_values_global, X_sample, feature_names=FEATURES, show=False)
plt.title('Importancia global de variables SHAP — Todo el dataset', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Cuota mensual y ratio DTI

Formula de amortizacion francesa (cuota constante):
```
C = L * r(1+r)^n / ((1+r)^n - 1)
```
donde L = capital, r = tipo mensual, n = meses

```
DTI = (Cuota + Otras deudas) / Ingresos * 100
```

In [ ]:
# Plazo: el cliente elige, pero se limita por edad (max 75 - age)
max_by_age = max(5, min(30, 75 - age))
tenure     = max(5, min(requested_years, max_by_age))
if tenure != requested_years:
    print(f"⚠ Plazo ajustado de {requested_years} a {tenure} años (limite por edad: 75 - {age} = {75-age})")

n       = tenure * 12
r       = max(rate, 0.5) / 100 / 12

if r > 0:
    cuota = loan_amount * r * (1+r)**n / ((1+r)**n - 1)
else:
    cuota = loan_amount / n

total_oblig = cuota + other_debts
dti = (total_oblig / income * 100) if income > 0 else 999

print(f"Capital: {loan_amount:,.0f} EUR | Tipo: {rate:.2f}% | Plazo: {tenure} anos")
print(f"Cuota mensual: {cuota:,.2f} EUR/mes")
print(f"Otras deudas:  {other_debts:,.2f} EUR/mes")
print(f"Total:         {total_oblig:,.2f} EUR/mes")
print(f"Ingresos:      {income:,.2f} EUR/mes")
print()
print(f">>> DTI: {dti:.1f}% <<<")


## 9. Decision bancaria — 3 niveles de DTI

| DTI | Decision |
|---|---|
| < 35% | **APROBADA** |
| 35-45% | **PENDIENTE ANALISTA DE RIESGOS** |
| > 45% | **DENEGADA** |

In [ ]:
dti_dec = 'approved' if dti < 35 else ('analyst' if dti <= 45 else 'denied')
auto_denial = None

if income == 0:
    dti_dec = 'denied'
    auto_denial = 'No has indicado ingresos.'
elif age > 65:
    dti_dec = 'denied'
    auto_denial = f'Con {age} anos el plazo no permite hipoteca estandar.'
elif not savings_ok:
    dti_dec = 'denied'
    auto_denial = (f'Ahorros netos insuficientes: necesitas {min_down:,.0f} EUR de entrada '
                   f'+ {total_taxes:,.0f} EUR de gastos = {min_down+total_taxes:,.0f} EUR total. '
                   f'Tienes {savings:,.0f} EUR.')

STATUS = {'approved':'HIPOTECA APROBADA','analyst':'PENDIENTE DE ANALISTA','denied':'HIPOTECA DENEGADA'}
print("=" * 52)
print(f"  {STATUS[dti_dec]}")
print("=" * 52)
if dti_dec == 'approved':
    print(f"  DTI {dti:.1f}% < 35% — dentro de parametros bancarios")
elif dti_dec == 'analyst':
    print(f"  Tu ratio deuda/ingresos ({dti:.1f}%) esta entre el 35% y el 45%.")
    print(f"  Esta operacion requiere revision por un analista de riesgos.")
    print(f"  Cuota mensual estimada: {cuota:,.0f} EUR.")
    print(f"  Otras deudas declaradas: {other_debts:,.0f} EUR/mes.")
else:
    print(f"  {auto_denial or f'DTI {dti:.1f}% supera el maximo del 45%.'}")

print()
print(f"  Precio vivienda: {estimated_price:,.0f} EUR")
print(f"  Capital financiado: {loan_amount:,.0f} EUR ({max_fin:.0f}%)")

## 10. Resultado final — Resumen completo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# ── Panel 1: Resumen del cliente y tipo ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')
fuente_str = 'PROYECCIONES BCE + SARIMA' if is_future else 'DATOS HISTORICOS'
resumen_text = (
    f"RESUMEN DE LA OPERACION\n"
    f"{'─'*40}\n"
    f"Provincia: {province} | CCAA: {ccaa}\n"
    f"Año: {year} ({fuente_str})\n"
    f"{'─'*40}\n"
    f"Precio vivienda:   {estimated_price:>10,.0f} EUR\n"
    f"Capital financiado:{loan_amount:>10,.0f} EUR ({max_fin:.0f}%)\n"
    f"Gastos compra:     {total_taxes:>10,.0f} EUR\n"
    f"{'─'*40}\n"
    f"TIPO OFRECIDO:     {rate:>10.2f}% TIN ({tipo})\n"
    f"Cuota mensual:     {cuota:>10,.0f} EUR\n"
    f"DTI:               {dti:>10.1f}%\n"
    f"Decision:          {STATUS[dti_dec]:>10}\n"
)
ax1.text(0.05, 0.95, resumen_text, transform=ax1.transAxes, fontsize=10,
         fontfamily='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#ecf0f1', alpha=0.8))
ax1.set_title('Datos de la operacion', fontweight='bold', fontsize=11)

# ── Panel 2: Composicion del tipo ──
ax2 = fig.add_subplot(gs[0, 1])
if tipo == 'variable':
    components = ['Euribor', 'Spread base', 'Ajuste perfil']
    values = [euribor, spread_base, profile_adj]
else:
    components = ['APRC', 'Ajuste perfil', 'Prima fija']
    values = [aprc, profile_adj, 0.50]

colors_bar = ['#3498db', '#e67e22', '#e74c3c' if profile_adj > 0 else '#27ae60']
ax2.barh(components, values, color=colors_bar, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.5)
for i, v in enumerate(values):
    s = '+' if v >= 0 else ''
    ax2.text(v + 0.02 if v >= 0 else v - 0.02, i, f'{s}{v:.3f}%',
             va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax2.set_xlabel('Porcentaje (%)')
ax2.set_title(f'Composicion del tipo {rate:.2f}% ({tipo})', fontweight='bold', fontsize=11)
ax2.grid(axis='x', alpha=0.3)

# ── Panel 3: SHAP contribuciones ──
ax3 = fig.add_subplot(gs[1, 0])
top_contribs = sorted(contribs_shap, key=lambda x: abs(x['shap']), reverse=True)[:6]
labels_shap = [c['var'] for c in top_contribs]
vals_shap   = [c['shap'] for c in top_contribs]
colors_shap = ['#c0392b' if v >= 0 else '#27ae60' for v in vals_shap]
ax3.barh(labels_shap, vals_shap, color=colors_shap, alpha=0.85, edgecolor='white')
ax3.axvline(0, color='black', linewidth=0.5)
for i, v in enumerate(vals_shap):
    s = '+' if v >= 0 else ''
    ax3.text(v + 0.003 if v >= 0 else v - 0.003, i, f'{s}{v:.4f}',
             va='center', ha='left' if v >= 0 else 'right', fontsize=8)
ax3.set_xlabel('SHAP (contribucion al spread)')
ax3.set_title('SHAP — ¿Por que este spread?', fontweight='bold', fontsize=11)
ax3.grid(axis='x', alpha=0.3)

# ── Panel 4: Posicion del cliente ──
ax4 = fig.add_subplot(gs[1, 1])
categories = ['Income ratio\n(ingresos/PIB_pc)', 'DTI (%)', 'Financiacion (%)']
client_vals = [income_ratio, dti, max_fin]

x_pos = range(len(categories))
bars = ax4.bar(x_pos, client_vals, color=['#27ae60' if dti < 35 else '#e67e22' if dti <= 45 else '#c0392b' for _ in client_vals],
               alpha=0.85, edgecolor='white', width=0.5)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(categories, fontsize=9)
for bar, val in zip(bars, client_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax4.set_title('Perfil del cliente', fontweight='bold', fontsize=11)
ax4.grid(axis='y', alpha=0.3)

fig.suptitle(f'Asesor Hipotecario — {province} ({year})', fontweight='bold', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print()
print("=" * 60)
print("  PRICING CON GRADIENT BOOSTING + SHAP")
print(f"  Modelo explicable — Cumple AI Act (Art. 13-14)")
print(f"  R² entrenamiento: {r2_train:.4f} (referencia)")
print(f"  RMSE CV temporal: {rmse_cv:.4f}% (validacion real)")
if is_future:
    print(f"  Fuente tipos: Proyecciones BCE ({year})")
    if tipo == 'variable':
        print(f"  Euribor: {euribor:.3f}% (BCE)")
    else:
        print(f"  APRC: {aprc:.3f}% = Euribor BCE ({euribor:.3f}%) + Diferencial SARIMAX ({dif_proj:.3f}%)" if is_future else "")
print("=" * 60)